In [12]:
# Making HTTP requests
import requests

# Building web interfaces
import gradio as gr

# Document Path
import os
from pathlib import Path

# Document Loading
from langchain_community.document_loaders import PyPDFLoader

# Text Splitting (Fixed the "s" at the end of splitters)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector Database
from langchain_chroma import Chroma

# Ollama Integration (For both Embeddings and the LLM)
from langchain_ollama import OllamaEmbeddings
from langchain_community.llms import Ollama

# RAG Logic and Prompting
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
print("You Have desired Library for running Rag Based application.")



You Have desired Library for running Rag Based application.


In [13]:
import requests

def check_ollama():
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print("Ollama is running on port 11434")
        else:
            print("Ollama responded but something is off")
    except Exception:
        print("Ollama is NOT running — start it with: ollama serve")

check_ollama()

Ollama is running on port 11434


In [ ]:
# Document Path
import os
from pathlib import Path

# Paths 
BASE_DIR      = Path(os.getcwd())        
CHROMA_DIR    = BASE_DIR / "chroma_db"
KB_FILE       = BASE_DIR / "KB_PCAI.pdf"

# Config
EMBED_MODEL   = "nomic-embed-text"   
LLM_MODEL     = "llama3"               
COLLECTION    = "AI_KB"
CHUNK_SIZE    = 300
CHUNK_OVERLAP = 50
TOP_K         = 3

# Verify paths exist
print(f"Base dir  : {BASE_DIR}")
print(f"Data file : {KB_FILE}")
print(f"Croma_DB folder exists: {CHROMA_DIR.exists()}")
print(f"Data file exists: {KB_FILE.exists()}")



Base dir  : c:\Users\kumaamar\OneDrive - Hewlett Packard Enterprise\New folder\AI_journey
Data file : c:\Users\kumaamar\OneDrive - Hewlett Packard Enterprise\New folder\AI_journey\KB_PCAI.pdf
Croma_DB folder exists: True
Data file exists: True


In [15]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loading the PDF
# Using str(KB_FILE) because PyPDFLoader expects a string path
loader = PyPDFLoader(str(KB_FILE))
raw_docs = loader.load()

print(f"Raw PDF pages loaded : {len(raw_docs)}")

# Initialize the Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " "], # Tries to split at paragraphs, then lines, then sentences
    add_start_index=True                 # Useful for tracking where the chunk sits in the doc
)

# Perform the splitting
chunks = splitter.split_documents(raw_docs)
print(f"Total chunks after splitting : {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")

# Verification
if len(chunks) > 0:
    print(f"\nSuccessfully split into {len(chunks)} chunks.")
else:
    print("\nNo text extracted.")

Raw PDF pages loaded : 3
Total chunks after splitting : 10

Sample chunk:
Machine learning is a subset of artificial intelligence that enables  
systems to learn and improve from experience without being explicitly  
programmed. 
 
Supervised learning uses labeled training data to learn a mapping from

Successfully split into 10 chunks.


In [16]:
from langchain_ollama import OllamaEmbeddings

# Initialize 
embedding_fn = OllamaEmbeddings(
    model=EMBED_MODEL 
)

print(f"Embedding model ready : {EMBED_MODEL}")

# Quick sanity test
try:
    test_vec = embedding_fn.embed_query("I am being tested Guys to be number")
    print(f"\nSuccess! Embedding dimension : {len(test_vec)}")
except Exception as e:
    print(f"\nModel Test Failed")


Embedding model ready : nomic-embed-text

Success! Embedding dimension : 768


In [6]:
from langchain_community.vectorstores import Chroma

# Create persistent ChromaDB vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_fn,
    collection_name=COLLECTION,
    persist_directory=str(CHROMA_DIR),
)

print(f"\nChromaDB collection '{COLLECTION}' ready")
print(f"\nTotal vectors stored : {vectorstore._collection.count()}")
print(f"\nPersisted at : {CHROMA_DIR}")


ChromaDB collection 'AI_KB' ready

Total vectors stored : 26

Persisted at : c:\Users\kumaamar\OneDrive - Hewlett Packard Enterprise\New folder\AI_journey\chroma_db


In [17]:
# On notebook restart, so you don't re-index every time

from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    collection_name=COLLECTION,
    embedding_function=embedding_fn,
    persist_directory=str(CHROMA_DIR),
)

print(f"ChromaDB loaded from : {CHROMA_DIR}")
print(f"\nTotal vectors: {vectorstore._collection.count()}")

ChromaDB loaded from : c:\Users\kumaamar\OneDrive - Hewlett Packard Enterprise\New folder\AI_journey\chroma_db

Total vectors: 26


In [18]:
# Set up the retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K},
)

print("Retriever ready")

Retriever ready


In [19]:
# Set up local LLM via Ollama
from langchain_community.llms import Ollama

llm = Ollama(
    model=LLM_MODEL,
    temperature=0.2,
)

print("LLM ready")

LLM ready


In [ ]:
# Building RAG Chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer using ONLY the context below.
If answer is not in context say: "I don't know."

CONTEXT: {context}
QUESTION: {question}
""")

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready")

RAG chain ready


In [20]:
import gradio as gr

def chat(question):
    answer = rag_chain.invoke(question)
    return answer

ui = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Ask a question"),
    outputs=gr.Textbox(label="Answer"),
    title="RAG Assistant ChatBot",
    description="Ask anything from the knowledge base. Beyond Knowledge Base, I dont have visibility"
)

ui.launch(inline=False, share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
